In [1]:
import jax.numpy as jnp
import qcsys as qs
import scqubits
import numpy as np

def transform_op_into_dressed_basis_jax(op_matrix, dressed_evecs):
    """
    Transform an operator into the dressed basis using JAX.

    Parameters:
    - op_matrix: A 2D JAX array representing the operator's matrix.
    - dressed_evecs: A 2D JAX array representing the dressed eigenvectors.

    Returns:
    - A 2D JAX array representing the transformed operator.
    """
    S = jnp.conj(dressed_evecs)
    data = jnp.dot(S, jnp.dot(op_matrix, S.T.conj()))
    return data

# fluxonium

In [2]:

qsf = qs.Fluxonium.create(
    25,
    {"Ej": 2.7, "Ec": 0.6, "El": 0.13, "phi_ext": 0.0},
    N_pre_diag=100,
)

scf = scqubits.Fluxonium(EJ=2.7,
                        EC=0.6,
                        EL=0.13,
                        flux=0,cutoff=100,
                        truncated_dim=25)


np.array_equal(np.array(scf.n_operator())[:25,:25] , np.array(qsf.ops['n'].data,dtype = 'complex128'))


2024-03-24 03:19:02.127794: E external/xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: UNKNOWN ERROR (100)


True

# transmon

In [3]:
qst_sc = qs.SingleChargeTransmon.create(
    N = 4,
    params = {"Ej": 40, "Ec": 0.5,"ng":0.0},
    N_max_charge=30
)

sct = scqubits.Transmon(
    EJ=40,
    EC=0.5,
    ng=0.0,
    ncut=30,
    truncated_dim = 4
    )

np.array_equal(np.array(sct.n_operator()),qst_sc.linear_ops['n'])

False

In [4]:
sct.n_operator().shape

(61, 61)

# hilbertspace

In [5]:
import jaxquantum as jqt
devices = [qsf, qst_sc]
f_indx = 0
t_indx = 1
Ns = [device.N for device in devices]

tn = qs.promote(qst_sc.common_ops()["n"], t_indx, Ns)
fn = qs.promote(qsf.ops["n"], f_indx, Ns)
g_tf = 0.2
system = qs.System.create(devices, couplings=[g_tf * tn @ fn])
system.params["g_tf"] = g_tf
Es, kets = system.calculate_eig_linear()


TypeError: incompatible dimensions [[25, 61], [25, 61]] and [[25, 4], [25, 4]]

In [7]:
hilbertspace = scqubits.HilbertSpace([scf, sct])
hilbertspace.add_interaction(g_strength=g_tf,op1=scf.n_operator, op2=sct.n_operator,add_hc=False)
evals, evecs = hilbertspace.hamiltonian().eigenstates()

In [15]:
import qutip as qt
isinstance(sct.n_operator(), qt.Qobj)

False

In [18]:
from scqubits.utils.spectrum_utils import *
operator = sct.n_operator()
subsystem = hilbertspace.subsystem_list[1]
subsys_list = hilbertspace.subsystem_list
evecs=None


In [31]:
dim = subsystem.truncated_dim

_, evecs = subsystem.eigensys(evals_count=dim)

state_table = evecs
states_in_columns = state_table

mtable = states_in_columns.conj().T @ operator @ states_in_columns
subsys_operator =  qt.Qobj(mtable)
subsys_operator

Quantum object: dims = [[4], [4]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[-6.01008263e-17 -1.23104130e+00 -1.47719213e-17  3.39734687e-02]
 [-1.23104130e+00 -2.06344709e-16 -1.70023304e+00  7.73715786e-17]
 [ 6.84948056e-17 -1.70023304e+00  9.82438489e-17  2.02624923e+00]
 [ 3.39734687e-02  8.38767850e-17  2.02624923e+00 -5.32346363e-16]]

In [53]:
subsystem.hamiltonian()

array([[1800.,  -20.,    0., ...,    0.,    0.,    0.],
       [ -20., 1682.,  -20., ...,    0.,    0.,    0.],
       [   0.,  -20., 1568., ...,    0.,    0.,    0.],
       ...,
       [   0.,    0.,    0., ..., 1568.,  -20.,    0.],
       [   0.,    0.,    0., ...,  -20., 1682.,  -20.],
       [   0.,    0.,    0., ...,    0.,  -20., 1800.]])

In [51]:
evals_count = 6
hamiltonian_mat = subsystem.hamiltonian()
evals, evecs = sp.linalg.eigh(
    hamiltonian_mat,
    eigvals_only=False,
    subset_by_index=(0, evals_count - 1),
    check_finite=False,
)
evals, evecs = order_eigensystem(evals, evecs)

In [48]:
hasattr(subsystem, "esys_method")

True

In [37]:
subsystem.truncated_dim

4

In [36]:
states_in_columns.shape

(61, 4)

In [35]:
operator.shape

(61, 61)

In [ ]:
operator_identitywrap_list = [
    qt.operators.qeye(the_subsys.truncated_dim) for the_subsys in subsys_list
]
subsystem_index = subsys_list.index(subsystem)
operator_identitywrap_list[subsystem_index] = subsys_operator


qt.tensor(operator_identitywrap_list)


In [9]:
evals[0]

-34.81858928744174

In [30]:
term = hilbertspace.interaction_list[0]
ops = term.id_wrap_all_ops(
            term.operator_list, hilbertspace.subsystem_list, bare_esys=None
        )

ops[0]

Quantum object: dims = [[25, 4], [25, 4]], shape = (100, 100), type = oper, isherm = True
Qobj data =
[[0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 ...
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]]

In [ ]:
operator_list = []
for term in self.interaction_list:
    if isinstance(term, qt.Qobj):
        operator_list.append(term)
    elif isinstance(term, (InteractionTerm, InteractionTermStr)):
        operator_list.append(
            term.hamiltonian(self.subsystem_list, bare_esys=bare_esys)
        )
    else:
        raise TypeError(
            "Expected an instance of InteractionTerm, InteractionTermStr, "
            "or Qobj; got {} instead.".format(type(term))
        )
hamiltonian = sum(operator_list)
return hamiltonian